# Notebook 72 — Final Equation Extraction, Paper Tables, and Report Bundle

This notebook converts the validated `structured_interaction` basis into:

- final governing equation
- coefficient table
- coefficient chart from regime metadata
- paper-ready model summary table
- final reconstruction figures
- Markdown report with figure/data references
- optional ZIP bundle of figures, CSV data, Markdown, and manifest

Pipeline:

```text
70 = final basis selected
71 = why structured interaction wins
72 = final equation + tables + report bundle
```

Final symbolic basis:

\[
\mathcal{B}_{SI}
=
[1,\; c,\; r,\; rc,\; c^3-\alpha c,\; r^2-\beta,\; rc^2]
\]

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os, glob, math, json, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, LeaveOneOut

try:
    from IPython.display import display, Markdown
except Exception:
    display = print
    Markdown = lambda x: x

np.random.seed(42)

OUTPUT_DIR = Path("paper72_outputs")
FIG_DIR = OUTPUT_DIR / "figures"
DATA_DIR = OUTPUT_DIR / "data"
REPORT_DIR = OUTPUT_DIR / "report"

for d in [OUTPUT_DIR, FIG_DIR, DATA_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True, "grid.alpha": 0.25})

## 1. Load data

In [ ]:
DATA_PATH = None

def autodetect_data_path():
    candidates = []
    patterns = [
        "*residual*flow*.parquet", "*residual*flow*.csv",
        "*governing*flow*.parquet", "*governing*flow*.csv",
        "*.parquet", "*.csv", "*.pkl", "*.pickle",
    ]
    for pat in patterns:
        candidates += glob.glob(pat)
        candidates += glob.glob(os.path.join("data", pat))
        candidates += glob.glob(os.path.join("outputs", pat))
    candidates = [c for c in candidates if os.path.isfile(c)]
    return candidates[0] if candidates else None

def synthetic_dataset(seed=42):
    rng = np.random.default_rng(seed)
    systems = ["entropy", "unevenness"]
    tasks = ["zeta_vs_gue", "zeta_vs_poisson"]
    forcing_modes = ["capacity_gap", "feature_gap", "condition_gap"]
    ks = [3, 5, 7]
    flow_modes = ["linear", "nonlinear"]

    rows = []
    sample_id = 0
    for system in systems:
        for task in tasks:
            for forcing_mode in forcing_modes:
                for k in ks:
                    for flow_mode in flow_modes:
                        c_grid = np.linspace(-1.25, 1.05, 42)
                        sys_shift = 0.06 if system == "entropy" else -0.04
                        task_shift = 0.05 if task == "zeta_vs_gue" else -0.03
                        force_shift = {"capacity_gap": 0.00, "feature_gap": 0.03, "condition_gap": 0.08}[forcing_mode]
                        k_shift = {3: -0.05, 5: 0.02, 7: 0.06}[k]
                        flow_shift = 0.05 if flow_mode == "nonlinear" else -0.02
                        nl_gain = 1.0 if flow_mode == "nonlinear" else 0.55

                        for path_id in range(14):
                            r = -0.75 + 0.10 * path_id + 0.05 * np.sin(0.7 * path_id)
                            for window_id, c in enumerate(c_grid):
                                g = (
                                    0.58 * np.tanh(1.35 * c)
                                    + 0.42 * c
                                    - 0.78 * r
                                    + 0.20 * r**2
                                    + nl_gain * 0.07 * c**2
                                    + nl_gain * 0.10 * r * c
                                    - nl_gain * 0.025 * r**3
                                    + sys_shift + task_shift + force_shift + k_shift + flow_shift
                                )
                                if forcing_mode == "condition_gap":
                                    g += 0.06 * np.sin(2.3 * c)
                                if system == "entropy":
                                    g += 0.03 * np.cos(1.2 * c)
                                if task == "zeta_vs_poisson":
                                    g -= 0.015 * c
                                if flow_mode == "linear":
                                    g -= 0.09 * r**2
                                    g += 0.015 * c * r

                                dc = c_grid[min(window_id + 1, len(c_grid)-1)] - c if window_id < len(c_grid)-1 else c - c_grid[max(window_id-1, 0)]
                                noise = 0.012 * rng.standard_normal()
                                next_r = r + (g + noise) * dc
                                rows.append({
                                    "system": system,
                                    "task": task,
                                    "forcing_mode": forcing_mode,
                                    "k": k,
                                    "flow_mode": flow_mode,
                                    "condition_coord": c,
                                    "residual": r,
                                    "predicted_flow": g + noise,
                                    "next_residual": next_r,
                                    "delta_condition": dc,
                                    "sample_id": sample_id,
                                    "path_id": path_id,
                                    "window_id": window_id,
                                })
                                r = next_r
                                sample_id += 1
    return pd.DataFrame(rows)

def load_dataframe(path):
    ext = os.path.splitext(str(path))[1].lower()
    if ext == ".parquet":
        return pd.read_parquet(path)
    if ext in [".pkl", ".pickle"]:
        return pd.read_pickle(path)
    return pd.read_csv(path)

def align_schema(df):
    alias_groups = {
        "condition_coord": ["condition_coord", "c", "condition", "cond", "condition_coordinate"],
        "residual": ["residual", "r", "resid"],
        "predicted_flow": ["predicted_flow", "flow", "g", "drdc", "delta_residual_per_condition"],
        "next_residual": ["next_residual", "r_next", "next_r"],
        "delta_condition": ["delta_condition", "dc", "delta_c"],
        "forcing_mode": ["forcing_mode", "forcing", "forcing_gap", "mode"],
    }
    rename = {}
    for canonical, aliases in alias_groups.items():
        for alias in aliases:
            if alias in df.columns and alias != canonical:
                rename[alias] = canonical
                break
    df = df.rename(columns=rename).copy()

    if "next_residual" not in df.columns and {"residual", "predicted_flow", "delta_condition"}.issubset(df.columns):
        df["next_residual"] = df["residual"] + df["predicted_flow"] * df["delta_condition"]

    if "delta_condition" not in df.columns and "condition_coord" in df.columns:
        df = df.sort_values(["condition_coord"]).copy()
        df["delta_condition"] = np.gradient(df["condition_coord"].to_numpy())

    defaults = {
        "system": "unknown_system",
        "task": "unknown_task",
        "forcing_mode": "unknown_forcing",
        "k": 5,
        "flow_mode": "unknown_flow",
        "sample_id": np.arange(len(df)),
        "path_id": 0,
        "window_id": np.arange(len(df)),
    }
    for key, value in defaults.items():
        if key not in df.columns:
            df[key] = value

    missing = [c for c in ["condition_coord", "residual", "predicted_flow"] if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns after alignment: {missing}")
    return df

if DATA_PATH is None:
    DATA_PATH = autodetect_data_path()

USE_SYNTHETIC_FALLBACK = DATA_PATH is None

if USE_SYNTHETIC_FALLBACK:
    df = synthetic_dataset()
    data_source = "synthetic_fallback"
else:
    df = align_schema(load_dataframe(DATA_PATH))
    data_source = DATA_PATH

df = align_schema(df)

print("Data source:", data_source)
print("Synthetic fallback:", USE_SYNTHETIC_FALLBACK)
print("Rows:", len(df))
display(df.head())

## 2. Final structured-interaction basis

In [ ]:
PRIMARY_BASIS = "structured_interaction"
FINAL_TERMS = ["1", "c", "r", "r c", "c^3_perp_c", "r^2_centered", "r c^2"]

def basis_stats(data):
    r = data["residual"].to_numpy(dtype=float)
    c = data["condition_coord"].to_numpy(dtype=float)
    return {
        "alpha_c3_on_c": float(np.sum(c**4) / max(np.sum(c**2), 1e-12)),
        "mean_r2": float(np.mean(r**2)),
    }

def structured_terms(data, stats=None):
    r = data["residual"].to_numpy(dtype=float)
    c = data["condition_coord"].to_numpy(dtype=float)
    if stats is None:
        stats = basis_stats(data)
    alpha = stats.get("alpha_c3_on_c", 0.0)
    beta = stats.get("mean_r2", 0.0)
    return {
        "1": np.ones_like(r),
        "c": c,
        "r": r,
        "r c": r * c,
        "c^3_perp_c": c**3 - alpha * c,
        "r^2_centered": r**2 - beta,
        "r c^2": r * c**2,
    }

def design_matrix(data, stats=None, columns=None):
    terms = structured_terms(data, stats=stats)
    if columns is None:
        columns = list(terms.keys())
    X = np.column_stack([terms.get(col, np.zeros(len(data))) for col in columns])
    return X, columns

def fit_template(sub, alpha=1e-6):
    stats = basis_stats(sub)
    X, names = design_matrix(sub, stats=stats, columns=FINAL_TERMS)
    y = sub["predicted_flow"].to_numpy(dtype=float)
    beta = np.linalg.solve(X.T @ X + alpha * np.eye(X.shape[1]), X.T @ y)
    pred = X @ beta
    row = {"n": len(sub), "template_rmse": math.sqrt(mean_squared_error(y, pred)), "basis_terms": "|".join(names)}
    for name, coef in zip(names, beta):
        row[f"coef_{name}"] = float(coef)
    return beta, pred, row, stats, names

def predict_with_beta(sub, beta, stats=None, columns=None):
    X, names = design_matrix(sub, stats=stats, columns=columns or FINAL_TERMS)
    return X @ np.asarray(beta, dtype=float)

def build_coef_table(data, min_rows=30):
    group_cols = ["system", "task", "forcing_mode", "k", "flow_mode"]
    rows, subsets, stats_map, names_map = [], {}, {}, {}
    for vals, sub in data.groupby(group_cols):
        if len(sub) < min_rows:
            continue
        beta, pred, stats_row, stats, names = fit_template(sub.copy())
        kval = int(vals[3]) if float(vals[3]).is_integer() else vals[3]
        regime_id = f"{vals[0]}|{vals[1]}|{vals[2]}|k={kval}|{vals[4]}"
        row = {
            "regime_id": regime_id,
            "system": vals[0],
            "task": vals[1],
            "forcing_mode": vals[2],
            "k": float(vals[3]),
            "flow_mode": vals[4],
        }
        row.update(stats_row)
        rows.append(row)
        subsets[regime_id] = sub.copy()
        stats_map[regime_id] = stats
        names_map[regime_id] = names
    coef_df = pd.DataFrame(rows).reset_index(drop=True)
    coef_cols = [c for c in coef_df.columns if c.startswith("coef_")]
    if coef_cols:
        coef_df[coef_cols] = coef_df[coef_cols].fillna(0.0)
    return coef_df, coef_cols, subsets, stats_map, names_map

coef_df, coef_cols, subsets, stats_map, names_map = build_coef_table(df)
display(coef_df.head())
coef_df.to_csv(DATA_DIR / "structured_interaction_coefficient_table.csv", index=False)

## 3. Global final equation fit

In [ ]:
global_stats = basis_stats(df)
X_global, global_terms = design_matrix(df, stats=global_stats, columns=FINAL_TERMS)
y_global = df["predicted_flow"].to_numpy(dtype=float)
beta_global = np.linalg.solve(X_global.T @ X_global + 1e-6 * np.eye(X_global.shape[1]), X_global.T @ y_global)
yhat_global = X_global @ beta_global
global_rmse = math.sqrt(mean_squared_error(y_global, yhat_global))

global_coef_table = pd.DataFrame({
    "term": global_terms,
    "coefficient": beta_global,
    "interpretation": [
        "intercept / baseline drift",
        "condition coordinate",
        "residual coordinate",
        "explicit residual-condition coupling",
        "cubic condition component with linear projection removed",
        "residual-square fluctuation around mean",
        "residual times squared condition",
    ],
})

display(global_coef_table)
print("Global field RMSE:", global_rmse)

global_coef_table.to_csv(DATA_DIR / "final_global_coefficients.csv", index=False)
df.assign(predicted_flow_structured_global=yhat_global).to_csv(DATA_DIR / "global_reconstruction_samples.csv", index=False)

## 4. Sparse metadata → coefficient chart

In [ ]:
def build_metadata_features(df_in, columns=None):
    X = pd.get_dummies(df_in[["forcing_mode", "flow_mode", "system", "task"]], drop_first=False)
    X["k"] = df_in["k"].astype(float).values
    X["k2"] = df_in["k"].astype(float).values ** 2
    ff = df_in["forcing_mode"].astype(str) + "__x__" + df_in["flow_mode"].astype(str)
    sf = df_in["system"].astype(str) + "__x__" + df_in["forcing_mode"].astype(str)
    X = pd.concat([
        X,
        pd.get_dummies(ff, prefix="ff"),
        pd.get_dummies(sf, prefix="sf"),
        pd.get_dummies(df_in["forcing_mode"].astype(str), prefix="fk").multiply(
            df_in["k"].astype(float).to_numpy().reshape(-1, 1)
        ),
    ], axis=1)
    if columns is not None:
        X = X.reindex(columns=columns, fill_value=0.0)
    return X.astype(float)

def term_stability_table(coef_df, coef_cols, n_splits=8, threshold=1e-4):
    n = len(coef_df)
    splitter = LeaveOneOut() if n <= 12 else KFold(n_splits=min(n_splits, n), shuffle=True, random_state=42)
    all_features = build_metadata_features(coef_df).columns.tolist()
    stability = {coef: {feat: 0 for feat in all_features} for coef in coef_cols}
    fold_count = 0
    for train_idx, _ in splitter.split(coef_df):
        train_df = coef_df.iloc[train_idx].reset_index(drop=True)
        X_train = build_metadata_features(train_df, columns=all_features)
        for coef in coef_cols:
            y = train_df[coef].to_numpy(dtype=float)
            scaler = StandardScaler()
            Xs = scaler.fit_transform(X_train)
            model = LassoCV(cv=min(5, len(train_df)), random_state=fold_count + 1, max_iter=30000)
            model.fit(Xs, y)
            for feat, val in zip(all_features, model.coef_):
                if abs(val) > threshold:
                    stability[coef][feat] += 1
        fold_count += 1
    return pd.DataFrame([
        {"coefficient": coef, "metadata_term": feat, "frequency": stability[coef][feat] / max(fold_count, 1),
         "count": stability[coef][feat], "folds": fold_count}
        for coef in coef_cols for feat in all_features
    ])

stability_df = term_stability_table(coef_df, coef_cols)
stable_terms = stability_df[stability_df["frequency"] >= 0.5].copy()

chart_rows = []
chart_matrix_rows = []
for coef in coef_cols:
    active_terms = stable_terms[stable_terms["coefficient"] == coef]["metadata_term"].tolist()
    if not active_terms:
        chart_rows.append({"coefficient": coef, "intercept": float(coef_df[coef].mean()), "active_metadata_terms": "", "n_active_terms": 0})
        continue
    X = build_metadata_features(coef_df)[active_terms]
    y = coef_df[coef].to_numpy(dtype=float)
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    model = Ridge(alpha=1.0)
    model.fit(Xs, y)
    chart_rows.append({"coefficient": coef, "intercept": float(model.intercept_), "active_metadata_terms": "|".join(active_terms), "n_active_terms": len(active_terms)})
    for term, weight in zip(active_terms, model.coef_):
        chart_matrix_rows.append({"coefficient": coef, "metadata_term": term, "weight_standardized": float(weight)})

chart_summary_df = pd.DataFrame(chart_rows)
chart_matrix_df = pd.DataFrame(chart_matrix_rows)

display(chart_summary_df)
display(chart_matrix_df.head())

stability_df.to_csv(DATA_DIR / "coefficient_chart_stability.csv", index=False)
stable_terms.to_csv(DATA_DIR / "coefficient_chart_stable_terms.csv", index=False)
chart_summary_df.to_csv(DATA_DIR / "final_coefficient_chart_summary.csv", index=False)
chart_matrix_df.to_csv(DATA_DIR / "final_coefficient_chart_weights.csv", index=False)

## 5. Final equation Markdown and LaTeX export

In [ ]:
TERM_LATEX = {
    "1": "1",
    "c": "c",
    "r": "r",
    "r c": "rc",
    "c^3_perp_c": "(c^3-\\alpha c)",
    "r^2_centered": "(r^2-\\beta)",
    "r c^2": "rc^2",
}

def equation_markdown(coef_table, precision=4):
    pieces = []
    for _, row in coef_table.iterrows():
        pieces.append(f"{row['coefficient']:.{precision}f}·{row['term']}")
    return "g(c,r) = " + " + ".join(pieces)

def equation_latex(coef_table, precision=3):
    pieces = []
    for _, row in coef_table.iterrows():
        term = TERM_LATEX.get(row["term"], row["term"])
        pieces.append(f"{row['coefficient']:.{precision}f}\\,{term}")
    return "g(c,r)=" + " + ".join(pieces)

eq_md = equation_markdown(global_coef_table)
eq_tex = equation_latex(global_coef_table)

final_equation_md = (
    "# Final Structured Interaction Equation\\n\\n"
    f"Global structured-interaction field RMSE: `{global_rmse:.6f}`\\n\\n"
    "```text\\n"
    f"{eq_md}\\n"
    "```\\n\\n"
    "LaTeX:\\n\\n"
    "```tex\\n"
    f"{eq_tex}\\n"
    "```\\n\\n"
    "Definitions:\\n\\n"
    "```tex\\n"
    "\\\\alpha = \\\\frac{\\\\sum c^4}{\\\\sum c^2},"
    "\\\\qquad"
    "\\\\beta = \\\\mathbb{E}[r^2]\\n"
    "```\\n"
)

(REPORT_DIR / "final_equation.md").write_text(final_equation_md, encoding="utf-8")
(REPORT_DIR / "final_equation.tex").write_text(eq_tex + "\\n", encoding="utf-8")

print(eq_md)
display(Markdown(f"$$ {eq_tex} $$"))

## 6. Paper-ready tables

In [ ]:
coef_mean_tables = {}
for group in ["forcing_mode", "flow_mode", "system", "task", "k"]:
    table = coef_df.groupby(group)[coef_cols].mean(numeric_only=True).reset_index()
    coef_mean_tables[group] = table
    table.to_csv(DATA_DIR / f"coefficient_means_by_{group}.csv", index=False)
    display(Markdown(f"### Coefficient means by `{group}`"))
    display(table)

model_table = pd.DataFrame([
    {
        "model": "structured_interaction",
        "role": "final paper-facing law",
        "n_terms": len(FINAL_TERMS),
        "field_rmse": global_rmse,
        "interpretation": "structured residual-flow coupling with partial polynomial decorrelation",
    }
])

display(model_table)
model_table.to_csv(DATA_DIR / "final_model_summary_table.csv", index=False)

## 7. Figures

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_df = df.copy()
plot_df["predicted_flow_structured_global"] = yhat_global
plot_df = plot_df.sort_values("condition_coord")
ax.scatter(plot_df["condition_coord"], plot_df["predicted_flow"], s=8, alpha=0.35, label="empirical / target")
ax.scatter(plot_df["condition_coord"], plot_df["predicted_flow_structured_global"], s=8, alpha=0.35, label="structured global")
ax.set_title("Final structured-interaction reconstruction")
ax.set_xlabel("condition coordinate c")
ax.set_ylabel("predicted flow g")
ax.legend()
plt.tight_layout()
fig.savefig(FIG_DIR / "figure1_global_reconstruction.png", dpi=220, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(global_coef_table["term"], global_coef_table["coefficient"])
ax.axhline(0, color="black", linewidth=1)
ax.set_title("Final global structured-interaction coefficients")
ax.set_ylabel("coefficient")
ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
fig.savefig(FIG_DIR / "figure2_final_global_coefficients.png", dpi=220, bbox_inches="tight")
plt.show()

if len(chart_matrix_df):
    heat = chart_matrix_df.pivot(index="coefficient", columns="metadata_term", values="weight_standardized").fillna(0.0)
    fig, ax = plt.subplots(figsize=(max(8, 0.35 * heat.shape[1]), 5))
    im = ax.imshow(heat.values, aspect="auto")
    ax.set_yticks(range(len(heat.index)))
    ax.set_yticklabels(heat.index)
    ax.set_xticks(range(len(heat.columns)))
    ax.set_xticklabels(heat.columns, rotation=65, ha="right", fontsize=8)
    ax.set_title("Final coefficient chart weights")
    fig.colorbar(im, ax=ax, label="standardized ridge weight")
    plt.tight_layout()
    fig.savefig(FIG_DIR / "figure3_coefficient_chart_heatmap.png", dpi=220, bbox_inches="tight")
    plt.show()
else:
    print("No stable metadata terms found for chart heatmap.")

rep_rid = coef_df.sort_values("template_rmse").iloc[len(coef_df)//2]["regime_id"]
rep_sub = subsets[rep_rid]
rep_stats = stats_map[rep_rid]
rep_beta = coef_df[coef_df["regime_id"] == rep_rid][coef_cols].iloc[0].to_numpy(dtype=float)
rep_terms = [c.replace("coef_", "") for c in coef_cols]
rep_pred = predict_with_beta(rep_sub, rep_beta, stats=rep_stats, columns=rep_terms)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sc0 = axes[0].scatter(rep_sub["condition_coord"], rep_sub["residual"], c=rep_sub["predicted_flow"], s=20)
axes[0].set_title("Representative empirical field")
axes[0].set_xlabel("c")
axes[0].set_ylabel("r")
fig.colorbar(sc0, ax=axes[0])

sc1 = axes[1].scatter(rep_sub["condition_coord"], rep_sub["residual"], c=rep_pred, s=20)
axes[1].set_title("Representative structured equation")
axes[1].set_xlabel("c")
axes[1].set_ylabel("r")
fig.colorbar(sc1, ax=axes[1])
plt.tight_layout()
fig.savefig(FIG_DIR / "figure4_representative_field_reconstruction.png", dpi=220, bbox_inches="tight")
plt.show()

## 8. Final Markdown report

In [ ]:
report_md = (
    "# Notebook 72 Report — Final Equation Extraction\\n\\n"
    "## Final basis\\n\\n"
    "```text\\n"
    f"{FINAL_TERMS}\\n"
    "```\\n\\n"
    "## Final global equation\\n\\n"
    "```text\\n"
    f"{eq_md}\\n"
    "```\\n\\n"
    "LaTeX:\\n\\n"
    "```tex\\n"
    f"{eq_tex}\\n"
    "```\\n\\n"
    "## Global fit\\n\\n"
    f"- Global field RMSE: `{global_rmse:.6f}`\\n"
    f"- Coefficient regimes: `{len(coef_df)}`\\n"
    f"- Data source: `{data_source}`\\n"
    f"- Synthetic fallback: `{USE_SYNTHETIC_FALLBACK}`\\n\\n"
    "## Final coefficient table\\n\\n"
    f"{global_coef_table.to_markdown(index=False)}\\n\\n"
    "## Model summary\\n\\n"
    f"{model_table.to_markdown(index=False)}\\n\\n"
    "## Figures\\n\\n"
    "- `figures/figure1_global_reconstruction.png`\\n"
    "- `figures/figure2_final_global_coefficients.png`\\n"
    "- `figures/figure3_coefficient_chart_heatmap.png`\\n"
    "- `figures/figure4_representative_field_reconstruction.png`\\n\\n"
    "## Data exports\\n\\n"
    "- `data/final_global_coefficients.csv`\\n"
    "- `data/structured_interaction_coefficient_table.csv`\\n"
    "- `data/final_coefficient_chart_summary.csv`\\n"
    "- `data/final_coefficient_chart_weights.csv`\\n"
    "- `data/final_model_summary_table.csv`\\n\\n"
    "## Final statement\\n\\n"
    "The final governing field is a structured interaction symbolic law. "
    "It preserves residual-condition coupling while reducing redundant raw polynomial alignment. "
    "Coefficients are interpretable and can be summarized globally or predicted from regime metadata through a sparse coefficient chart.\\n"
)

(REPORT_DIR / "notebook72_report.md").write_text(report_md, encoding="utf-8")
display(Markdown(report_md[:2200] + "\\n\\n..."))

## 9. Manifest and optional ZIP bundle

In [ ]:
manifest = {
    "notebook": "72_final_equation_extraction_and_paper_tables",
    "data_source": data_source,
    "synthetic_fallback": bool(USE_SYNTHETIC_FALLBACK),
    "primary_basis": PRIMARY_BASIS,
    "final_terms": FINAL_TERMS,
    "global_field_rmse": float(global_rmse),
    "n_regimes": int(len(coef_df)),
    "outputs": {
        "figures": sorted([p.name for p in FIG_DIR.glob("*")]),
        "data": sorted([p.name for p in DATA_DIR.glob("*")]),
        "report": sorted([p.name for p in REPORT_DIR.glob("*")]),
    },
}

(OUTPUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
display(pd.DataFrame({
    "folder": ["figures", "data", "report"],
    "n_files": [
        len(manifest["outputs"]["figures"]),
        len(manifest["outputs"]["data"]),
        len(manifest["outputs"]["report"]),
    ],
}))

print("Saved outputs under:", OUTPUT_DIR)

In [ ]:
# OPTIONAL ZIP EXPORT / DOWNLOAD
# Uncomment this cell in Colab when you want a single bundle of figures, data, Markdown report, and manifest.

# ZIP_PATH = "paper72_outputs_bundle.zip"
# with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
#     for p in OUTPUT_DIR.rglob("*"):
#         if p.is_file():
#             z.write(p, p.relative_to(OUTPUT_DIR.parent))
#
# print("Created:", ZIP_PATH)
#
# # In Google Colab, uncomment:
# # from google.colab import files
# # files.download(ZIP_PATH)

## Final statement

Notebook 72 is the clean paper endpoint.

It exports:

- final equation as Markdown and LaTeX
- global coefficient table
- coefficient chart tables
- model summary table
- reconstruction figures
- Markdown report
- optional ZIP bundle cell

Next choices:

1. write `paper.tex` v1 from 64–72 outputs  
2. generate repo README with paper figures  
3. refactor 64–72 to import from `src/zeta_constraint`